In [2]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")
base_url = "https://api.deepseek.com/v1"

api_key

'sk-ebfb06c810a1412fa88b260afe7d5d46'

In [3]:
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model="deepseek-chat",    
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)
llm

ChatDeepSeek(client=<openai.resources.chat.completions.completions.Completions object at 0x763642d01100>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x763642d674a0>, root_client=<openai.OpenAI object at 0x763643657500>, root_async_client=<openai.AsyncOpenAI object at 0x763642da62d0>, model_name='deepseek-chat', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), max_retries=2, api_key=SecretStr('**********'), api_base='https://api.deepseek.com/v1')

In [8]:
from typing import Any

from langchain.agents import AgentState, create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import Runtime, before_model

# @before_model   
# def before_model_print(state: AgentState, runtime: Runtime) ->dict[str, Any] | None:
#     print(f" state: {state}")
#     print(f" runtime: {runtime}")
#     messages = state["messages"]
#     context = runtime.context
#     print(f" messages: {messages}")
#     print(f" context: {context}")
#     return {
#         "messages": state["messages"]
#     }


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"
agent = create_agent(
    model=llm,
    tools=[get_weather],
    # middleware=[before_model_print],
    system_prompt="You are a helpful assistant",
)
human_message = HumanMessage(content="What's the weather in New York?")
response = agent.stream({
    "messages": [human_message]
},stream_mode="values")
# define message as a list of dict[str,any]
messages: list[dict[str, Any]] = []
for msg in response:
    messages.append(msg)


In [ ]:

# for msg in messages:
#     print("Human:", msg)
from langchain_core.messages import ToolMessage
from langchain_core.messages import AIMessage
last_message = messages[-1]['messages']
for msg in last_message:
    # print(type(msg))
    if (type(msg) == HumanMessage):
        print("Human:", msg.content)
    elif (type(msg) == AIMessage):
        print("AI:", msg)
    elif (type(msg) == ToolMessage):
        print("Tool:", msg)
    
# last_message

<class 'langchain_core.messages.human.HumanMessage'>
Human: content="What's the weather in New York?" additional_kwargs={} response_metadata={} id='8b25ccb1-83cb-470f-ad61-8252ab38b378'
<class 'langchain_core.messages.ai.AIMessage'>
AI: content="I'll check the weather in New York for you." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 312, 'total_tokens': 367, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 56}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': '7bd90a35-ee7d-4293-88a2-f438e7010552', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019d8202-50ab-7642-8ec4-7e5d6232288c-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'call_00_9RFLpSAOXF7qc5vZuArFz36N', 't